In [2]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


### Langchain Integration
https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html#langchain-integration

#### Harmonized Model Initialization
The init_llm and init_embedding_model functions allow easy initialization of langchain model interfaces in a harmonized way in generative AI hub sdk

In [15]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from gen_ai_hub.proxy.langchain.init_models import init_llm

template = """Question: {question}
    Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=['question'])
question = 'What is a supernova?'

llm = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({'question': question})
print(response)

A supernova is a powerful and luminous explosion that occurs at the end of a star's life cycle. Let's break down the process step by step:

1. **Star Lifecycle**: Stars are massive celestial bodies composed primarily of hydrogen and helium. Throughout their lives, they undergo nuclear fusion, converting hydrogen into helium and releasing energy in the process.

2. **Fuel Depletion**: As a star exhausts its nuclear fuel, it can no longer sustain the fusion reactions that counteract the gravitational forces trying to collapse the star.

3. **Core Collapse**: In massive stars, the core collapses under gravity once the nuclear fuel is depleted. This collapse happens rapidly, causing the outer layers of the star to fall inward.

4. **Explosion**: The collapsing core rebounds off the dense core material, creating shock waves that propel the outer layers into space. This results in a supernova explosion.

5. **Types of Supernovae**: There are different types of supernovae, primarily classifie

init_embedding_model

In [5]:
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

text = 'Every decoding is another encoding.'

embeddings = init_embedding_model('text-embedding-3-large')
response = embeddings.embed_query(text)
print(response)


[-0.02279829792678356, 0.016395503655076027, -0.02253410406410694, -0.015330961905419827, -0.010241362266242504, 0.0040911054238677025, 0.0022980901412665844, 0.052900753915309906, 0.000381719961296767, 0.0036171122919768095, 0.05283858999609947, 0.0018474080134183168, -0.0253159012645483, 0.019550278782844543, 0.04867366701364517, 0.01384681835770607, 0.005361562594771385, -0.008197751827538013, -0.004234857391566038, -0.015439746901392937, 0.007867510430514812, -0.04892231896519661, -0.029713936150074005, -0.01493467204272747, -0.016084687784314156, 0.02031177468597889, -0.0041998908855021, -0.03013353794813156, -0.0054431515745818615, -0.0054897740483284, -0.013862359337508678, -0.0007012768764980137, 0.00013865272921975702, -0.019301626831293106, 0.004164924379438162, 0.011313674971461296, 0.00965858343988657, -0.022471942007541656, -0.005738426465541124, 0.01031129527837038, 0.021321924403309822, 0.02495846338570118, -0.03125247359275818, 0.07297942042350769, 0.00941770151257515, 

#### Chat model

In [35]:
from langchain_core.prompts.chat import (
    AIMessagePromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

chat_llm = ChatOpenAI(proxy_model_name='gpt-4o', proxy_client=proxy_client)

template = 'You are a helpful assistant that translates english to Chinese.'
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

example_human = HumanMessagePromptTemplate.from_template('Hi')
example_ai = AIMessagePromptTemplate.from_template('Ahoy!')
human_template = '{text}'

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)
chat_prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, example_human, example_ai, human_message_prompt])

chain = chat_prompt | chat_llm

response = chain.invoke({'text': 'I love planking.'})
print(response.content)


我喜欢平板支撑。


#### Structured model outputs

In [38]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.prompts.chat import HumanMessage
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
chat_model = ChatOpenAI(proxy_model_name="gpt-4o", proxy_client=get_proxy_client())
chat_model = chat_model.with_structured_output(method="json_schema", schema=Person, strict=True)

message = HumanMessage(content="Tell me about a person named John who is 30")
print(chat_model.invoke([message]))


name='John' age=30


### Agent

#### Define tools

In [85]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Define agents

In [86]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [87]:

result =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "What is weather in Shanghai?"
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='5d82fe8b-65d3-4958-bf91-4be7280024c5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_MYWe6Uf8BQACwTjp9ZSAEdzE', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CnKCYkbK4rWGMCjetUakOeX8tqmz8', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2628-bad4-7d22-9cb6-c045e247f8ed-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'c

In the below case the tool "search" is not applied

In [88]:

result1 =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "Where is the location of Shanghai"
            }
        ]
    }
)
print(result1)

{'messages': [HumanMessage(content='Where is the location of Shanghai', additional_kwargs={}, response_metadata={}, id='b8ab850c-e0ba-4521-9191-5b779efb5c70'), AIMessage(content='Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 78, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CnKCefjN5V1dZlLE3pyq2dfOZjB7C', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b2628-d055-7a90-9563-0526d9138050-0', usage_metadata={'input_tokens': 78, 'output_tokens': 39, 'total_token

To let agent to use the tool, change the question closer to the tool description.

In [73]:

result2 =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "Search for the location of Shanghai"
            }
        ]
    }
)
print(result2)

{'messages': [HumanMessage(content='Search for the location of Shanghai', additional_kwargs={}, response_metadata={}, id='eae86236-be99-4540-afd6-d4c619b6cbb6'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_UMMMRgA1E8U3GIVLdEpIMMeQ', 'function': {'arguments': '{"query":"location of Shanghai"}', 'name': 'search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 78, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CnJung1BZ063cA4D4rdlxGvd6QDIV', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2617-ef3d-7173-8789-73072a687e65-0', tool_calls=[{'name': 'search', 'args': {'query': 'location of Shang

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt



agent = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)




In [79]:
# The system prompt will be set dynamically based on context
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for the explaination of machine learning"}]},
    context={"user_role": "beginner"}
)
print(result1)

{'messages': [HumanMessage(content='Search for the explaination of machine learning', additional_kwargs={}, response_metadata={}, id='8a1b7425-e24f-4a80-8105-b229b7d82488'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bzpWMXoY0m6HAAHXIeasY6Nc', 'function': {'arguments': '{"query":"explanation of machine learning"}', 'name': 'search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 61, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CnK6hkCqpRJ9UrYqx4jXBZlJOEy35', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2623-30c5-7512-ba76-8ed8cea8dc41-0', tool_calls=[{'name': 'search', 'args': {'que

In [80]:
# The system prompt will be set dynamically based on context
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for the explaination of machine learning"}]},
    context={"user_role": "expert"}
)
print(result2)

{'messages': [HumanMessage(content='Search for the explaination of machine learning', additional_kwargs={}, response_metadata={}, id='f16726c2-1c61-4427-bd9c-50b66a77b120'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_qmBTIN9AbNwRzruZReOWLoPS', 'function': {'arguments': '{"query":"explanation of machine learning"}', 'name': 'search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 59, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CnK6ppS2VHRcAhTNlFaONzuX0knDe', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2623-4ea1-70f3-a865-3b24c17e4774-0', tool_calls=[{'name': 'search', 'args': {'que

In [44]:
from langchain.agents import create_agent
from langchain.tools import tool

# 1. Define a tool
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's 25 degrees Celsius in {city}."

# 2. Initialize your model
# llm = init_llm(
#     'gpt-4o', 
#     temperature=0.1,
#     max_tokens=300
# )

# 3. Create the agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when necessary."
)

# 4. Invoke the agent (agents take a list of messages as input)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Tokyo?"}]}
)

print(result)


{'messages': [HumanMessage(content='What is the weather in Tokyo?', additional_kwargs={}, response_metadata={}, id='41b84c65-43a0-48df-aa57-5d58190d2ff3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PN8fNaIXVLMsXL4toFRIeiPG', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 62, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CmwxOWRiwrVRBpDpW0bYPdFWaTY2H', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--66de4fb8-a464-42e2-b910-10b93a0868c8-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_PN8fNaIXV

In [50]:
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

web_search


In [52]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model=llm,
    tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

NameError: name 'web_search' is not defined

In [56]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"




from langchain.agents.middleware import wrap_tool_call
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )    


 


agent = create_agent(
    model=llm,
    tools=[search, get_weather],
    middleware=[handle_tool_errors] 
)


In [59]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

literary_agent = create_agent(
    model=llm,
    system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked with analyzing literary works.",
            },
            {
                "type": "text",
                "text": "<the entire contents of '三国演义'>",
                "cache_control": {"type": "ephemeral"}
            }
        ]
    )
)

result = literary_agent.invoke(
    {"messages": [HumanMessage("Analyze the major themes in '三国演义'.")]}
)

In [60]:
print(result)

{'messages': [HumanMessage(content="Analyze the major themes in '三国演义'.", additional_kwargs={}, response_metadata={}, id='dfd3a381-0cb9-442b-a48f-0980b78a2697'), AIMessage(content='"三国演义" (Romance of the Three Kingdoms) is a classic Chinese historical novel attributed to Luo Guanzhong, written in the 14th century. It is one of the Four Great Classical Novels of Chinese literature and is renowned for its complex narrative and rich character development. The novel covers the turbulent period at the end of the Han Dynasty and the Three Kingdoms era, focusing on the power struggles between the states of Wei, Shu, and Wu. Here are some of the major themes in "三国演义":\n\n1. **Loyalty and Betrayal**: Loyalty is a central theme, exemplified by the relationships between lords and their retainers. Characters like Guan Yu and Zhang Fei are celebrated for their unwavering loyalty to Liu Bei. Conversely, betrayal is also prevalent, with figures like Lü Bu and Cao Cao often depicted as treacherous, h